In [ ]:
import pandas as pd
import numpy as np
import torch
import joblib
from preprocess import prepare_base_df, add_lags_and_rollings, split_by_day

# 基本參數
SEQ_LEN = 8
DATA_PATH = "data/processed/sapporo_density.parquet" 

# 載入資料
df = prepare_base_df(DATA_PATH)
df = add_lags_and_rollings(df, seq_len=SEQ_LEN)
train_df, val_df, test_df = split_by_day(df)

In [ ]:
from LGBM import LGBMModel
lgbm_trainer = LGBMModel(n_estimators=500)
print("正在訓練 LGBM...")
lgbm_trainer.train(train_df, val_df, SEQ_LEN)
joblib.dump(lgbm_trainer, "models/lgbm_best.joblib")
print("LGBM 儲存完成。")

In [ ]:
from GRU import GRURegEmbed
from LSTM import SeqDatasetEmbed
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gru = GRURegEmbed(hidden=64).to(device)
train_loader = DataLoader(SeqDatasetEmbed(train_df, SEQ_LEN), batch_size=256, shuffle=True)
optimizer = torch.optim.Adam(model_gru.parameters(), lr=0.001)

print("正在訓練 GRU...")
model_gru.train()
for epoch in range(5):
    for batch in train_loader:
        xs, wd, tid, xid, yid, wknd, target = [b.to(device) for b in batch]
        optimizer.zero_grad()
        loss = torch.nn.MSELoss()(model_gru(xs, wd, tid, xid, yid, wknd), target)
        loss.backward()
        optimizer.step()
torch.save({"state_dict": model_gru.state_dict(), "config": {"hidden": 64}}, "models/gru_best.pkl")
print("GRU 儲存完成。")

In [ ]:
from PredRNN import PredRNNRegEmbed
from ConvLSTM import PatchDatasetEmbed

model_pr = PredRNNRegEmbed(hid_ch=64).to(device)
# PredRNN 使用 Patch 數據，計算較重，batch_size 設小一點
train_loader_patch = DataLoader(PatchDatasetEmbed(train_df, SEQ_LEN, patch_radius=4), batch_size=64, shuffle=True)
optimizer_pr = torch.optim.Adam(model_pr.parameters(), lr=0.001)

print("正在訓練 PredRNN...")
model_pr.train()
for epoch in range(3):
    for batch in train_loader_patch:
        patches, wd, tid, xid, yid, wknd, target = [b.to(device) for b in batch]
        optimizer_pr.zero_grad()
        loss = torch.nn.MSELoss()(model_pr(patches, wd, tid, xid, yid, wknd), target)
        loss.backward()
        optimizer_pr.step()
torch.save({"state_dict": model_pr.state_dict(), "config": {"hid_ch": 64}}, "models/predrnn_best.pkl")
print("PredRNN 儲存完成。")

In [ ]:
import torch
import joblib
from ConvLSTM import load_convlstm_embed
from LSTM import load_lstm_embed
from GRU import GRURegEmbed
from PredRNN import PredRNNRegEmbed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. 載入原本的兩個 (使用 load 函式)
model_cl, cfg_cl = load_convlstm_embed("models/convlstm_best.pkl")
model_ls, cfg_ls = load_lstm_embed("models/lstm_best.pkl")

# 2. 載入剛訓練好的三個 (手動加載 state_dict)
model_lgbm = joblib.load("models/lgbm_best.joblib")

# GRU
ckpt_gru = torch.load("models/gru_best.pkl", map_location=device)
model_gru = GRURegEmbed(hidden=ckpt_gru["config"]["hidden"]).to(device)
model_gru.load_state_dict(ckpt_gru["state_dict"])
model_gru.eval()

# PredRNN
ckpt_pr = torch.load("models/predrnn_best.pkl", map_location=device)
model_pr = PredRNNRegEmbed(hid_ch=ckpt_pr["config"]["hid_ch"]).to(device)
model_pr.load_state_dict(ckpt_pr["state_dict"])
model_pr.eval()

print("所有模型載入成功！")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import root_mean_squared_error
from ConvLSTM import predict_convlstm_embed
from LSTM import predict_lstm_embed

# 取得預測值
print("正在執行各模型預測...")
y_true = test_df["count"].values

# 呼叫預測函式
y_pred_lgbm = model_lgbm.predict(test_df, SEQ_LEN)
y_pred_lstm = predict_lstm_embed(model_ls, test_df, SEQ_LEN)
y_pred_cl   = predict_convlstm_embed(model_cl, test_df, SEQ_LEN)

# 因為 GRU 跟 LSTM 結構一樣，可以直接套用 LSTM 的預測邏輯 (假設 preprocess 方式相同)
y_pred_gru  = predict_lstm_embed(model_gru, test_df, SEQ_LEN)

# 因為 PredRNN 跟 ConvLSTM 資料格式(Patch)一樣，套用 ConvLSTM 預測邏輯
y_pred_pr   = predict_convlstm_embed(model_pr, test_df, SEQ_LEN)

preds = {
    "LightGBM": y_pred_lgbm,
    "LSTM": y_pred_lstm,
    "GRU": y_pred_gru,
    "ConvLSTM": y_pred_cl,
    "PredRNN": y_pred_pr
}

# 1. 繪製 RMSE 比較圖
res = [{"Model": k, "RMSE": root_mean_squared_error(y_true, v)} for k, v in preds.items()]
df_res = pd.DataFrame(res).sort_values("RMSE")

plt.figure(figsize=(10, 6))
sns.barplot(x="Model", y="RMSE", data=df_res, palette="viridis")
plt.title("Final Model Comparison (Test Set RMSE)")
plt.ylabel("RMSE (Original Scale)")
plt.show()

# 2. 繪製 24 小時人流預測對比圖 (選取測試集中的一個網格)
sample_mask = (test_df['x'] == 10) & (test_df['y'] == 10) # 換成你感興趣的座標
target_df = test_df[sample_mask].head(48) # 觀察 48 個時段

plt.figure(figsize=(15, 6))
plt.plot(target_df['t'].values, target_df['count'].values, label='Actual', color='black', lw=3)
for name, y_val in preds.items():
    plt.plot(target_df['t'].values, y_val[sample_mask][:48], label=name, alpha=0.8)

plt.xlabel("Time Slot (t)")
plt.ylabel("People Count")
plt.title("24H Flow Prediction Comparison at Grid (10,10)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()